# Phase 2 — Measure
## 01 — Stock-on-Hand Reshape

### Objective
Transform the raw Stock on Hand report from wide store-level format into a governed long-format inventory dataset suitable for SQL, Power BI, inventory analysis, and downstream AI components.

### Source
- `Stock on Hand Report.xlsx`

### Target Structure
`Category | Model | Description | Store | Quantity`

### Validation Requirements
- Preserve all source products.
- Preserve all store-level stock quantities.
- Reconcile reshaped quantities against the source `Total` column.
- Identify missing, invalid, or unexpected inventory values.
- Do not silently modify source values.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find project root
current_path = Path.cwd()

if current_path.name == "Phase_2_Measure":
    PROJECT_ROOT = current_path.parent.parent
elif current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

Project root : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System
Raw data dir : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\raw
Processed dir: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed


In [2]:
## Locate the source file
stock_files = list(RAW_DIR.glob("*Stock*Hand*.xlsx"))

print(f"Matching files found: {len(stock_files)}")

for file in stock_files:
    print("-", file.name)

Matching files found: 1
- Stock on Hand Report.xlsx


In [3]:
if len(stock_files) != 1:
    raise ValueError(
        f"Expected exactly 1 Stock-on-Hand workbook, found {len(stock_files)}"
    )

STOCK_FILE = stock_files[0]

excel_file = pd.ExcelFile(STOCK_FILE)

print("Workbook:", STOCK_FILE.name)
print("Sheets:", excel_file.sheet_names)

Workbook: Stock on Hand Report.xlsx
Sheets: ['Sheet1']


### Step 1— Source Structure Validation.

In [4]:
# Load source stock data
stock_raw = pd.read_excel(
    STOCK_FILE,
    sheet_name="Sheet1"
)

print("Shape:", stock_raw.shape)
print("\nColumns:")
print(stock_raw.columns.tolist())

display(stock_raw.head(10))

Shape: (383, 11)

Columns:
['Category', 'Model', 'Description', 'Gorey', 'Dundrum', 'Cavan', 'Navan', 'Blanch', 'Sandyford', 'Belfast', 'Total']


,Category,Model,Description,Gorey,Dundrum,Cavan,Navan,Blanch,Sandyford,Belfast,Total
0,COOKER HOODS,ERACIXA60,Elica 52cm Canopy Hood for 60cm units,71,0,0,4,1,0,0,76
1,COOKER HOODS,P2152CH,Powerpoint 52cm Canopy Hood,5,0,4,0,3,0,0,12
2,COOKER HOODS,INTEGRATA60,Elica 5414601 60cm Integrata Integrated Hood Grey,5,0,2,1,1,0,1,10
3,COOKER HOODS,DGE5861HM,AEG 80cm Canopy Cooker Hood,2,0,0,0,2,1,2,7
4,COOKER HOODS,P2110XBSS,Powerpoint 60cm SS Traditional Hood,2,0,1,1,2,0,1,7
5,COOKER HOODS,HLTHDS110SL/,RANGEMASTER 110CM COOKER HOOD,1,0,1,1,1,1,1,6
6,COOKERS,LKR555100X,Electrolux 55cm St/St Cooker,10,0,6,1,3,1,2,23
7,COOKERS,LKR555100B,Electrolux 55cm Black Cooker,8,0,2,1,4,1,2,18
8,COOKERS,LKR655210X,Electrolux 60cm St/St Cooker,4,1,4,1,4,1,3,18
9,COOKERS,LKI655200X,Electrolux 60cm St/St Induction Cooker,6,1,2,1,2,1,2,15


In [5]:
print("Data types:")
display(stock_raw.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    stock_raw.isna()
    .sum()
    .to_frame("missing_count")
    .sort_values("missing_count", ascending=False)
)

print("\nDuplicate full rows:", stock_raw.duplicated().sum())

Data types:


,dtype
Category,object
Model,object
Description,object
Gorey,int64
Dundrum,int64
Cavan,int64
Navan,int64
Blanch,int64
Sandyford,int64
Belfast,int64



Missing values:


,missing_count
Category,0
Model,0
Description,0
Gorey,0
Dundrum,0
Cavan,0
Navan,0
Blanch,0
Sandyford,0
Belfast,0



Duplicate full rows: 0


In [6]:
expected_id_columns = [
    "Category",
    "Model",
    "Description"
]

expected_store_columns = [
    "Gorey",
    "Dundrum",
    "Cavan",
    "Navan",
    "Blanch",
    "Sandyford",
    "Belfast"
]

expected_total_column = "Total"

print("Expected ID columns present:")
for col in expected_id_columns:
    print(f"{col}: {col in stock_raw.columns}")

print("\nExpected store columns present:")
for col in expected_store_columns:
    print(f"{col}: {col in stock_raw.columns}")

print(
    "\nTotal column present:",
    expected_total_column in stock_raw.columns
)

Expected ID columns present:
Category: True
Model: True
Description: True

Expected store columns present:
Gorey: True
Dundrum: True
Cavan: True
Navan: True
Blanch: True
Sandyford: True
Belfast: True

Total column present: True


### Step 2 — Inventory Value Profiling & Source Reconciliation

In [7]:
## Duplicate and Model-key validation
# Full-row duplicates
full_duplicates = stock_raw.duplicated().sum()

# Duplicate Model identifiers
duplicate_models = stock_raw["Model"].duplicated().sum()

# Unique models
unique_models = stock_raw["Model"].nunique()

print("Total rows             :", len(stock_raw))
print("Unique models          :", unique_models)
print("Duplicate full rows    :", full_duplicates)
print("Duplicate Model values :", duplicate_models)

Total rows             : 383
Unique models          : 383
Duplicate full rows    : 0
Duplicate Model values : 0


In [8]:
## Profile store quantities
store_profile = stock_raw[expected_store_columns].agg(
    ["count", "min", "max", "sum"]
).T

store_profile.index.name = "Store"

display(store_profile)

,count,min,max,sum
Store,,,,
Gorey,383,-2,145,2850
Dundrum,383,0,7,293
Cavan,383,0,21,1094
Navan,383,-1,5,234
Blanch,383,-1,12,871
Sandyford,383,-1,6,254
Belfast,383,0,6,405


In [9]:
## Check for negative stock
negative_counts = (stock_raw[expected_store_columns] < 0).sum()

negative_summary = pd.DataFrame({
    "negative_count": negative_counts
})

display(negative_summary)

print(
    "\nTotal negative store-level values:",
    negative_counts.sum()
)

,negative_count
Gorey,2
Dundrum,0
Cavan,0
Navan,6
Blanch,3
Sandyford,1
Belfast,0



Total negative store-level values: 12


In [10]:
## Row-level reconciliation

stock_validation = stock_raw.copy()

stock_validation["Calculated_Total"] = (
    stock_validation[expected_store_columns].sum(axis=1)
)

stock_validation["Total_Difference"] = (
    stock_validation["Calculated_Total"]
    - stock_validation["Total"]
)

mismatch_count = (
    stock_validation["Total_Difference"] != 0
).sum()

print("Rows checked       :", len(stock_validation))
print("Matching totals    :", len(stock_validation) - mismatch_count)
print("Mismatched totals  :", mismatch_count)

display(
    stock_validation.loc[
        stock_validation["Total_Difference"] != 0,
        [
            "Category",
            "Model",
            "Description",
            "Total",
            "Calculated_Total",
            "Total_Difference"
        ]
    ]
)


Rows checked       : 383
Matching totals    : 383
Mismatched totals  : 0


,Category,Model,Description,Total,Calculated_Total,Total_Difference


In [11]:
## Grand-total reconciliation

source_grand_total = stock_raw["Total"].sum()

calculated_grand_total = (
    stock_raw[expected_store_columns]
    .sum()
    .sum()
)

difference = calculated_grand_total - source_grand_total

print("Source Total column :", source_grand_total)
print("Sum of store stock  :", calculated_grand_total)
print("Difference          :", difference)
print("Reconciled          :", difference == 0)

Source Total column : 6001
Sum of store stock  : 6001
Difference          : 0
Reconciled          : True


#### Step 2.1 — Wide → Long Reshape

In [12]:
stock_long = stock_raw.melt(
    id_vars=[
        "Category",
        "Model",
        "Description"
    ],
    value_vars=expected_store_columns,
    var_name="Store",
    value_name="Quantity"
)

print("Original shape :", stock_raw.shape)
print("Long shape     :", stock_long.shape)

display(stock_long.head(15))

Original shape : (383, 11)
Long shape     : (2681, 5)


,Category,Model,Description,Store,Quantity
0,COOKER HOODS,ERACIXA60,Elica 52cm Canopy Hood for 60cm units,Gorey,71
1,COOKER HOODS,P2152CH,Powerpoint 52cm Canopy Hood,Gorey,5
2,COOKER HOODS,INTEGRATA60,Elica 5414601 60cm Integrata Integrated Hood Grey,Gorey,5
3,COOKER HOODS,DGE5861HM,AEG 80cm Canopy Cooker Hood,Gorey,2
4,COOKER HOODS,P2110XBSS,Powerpoint 60cm SS Traditional Hood,Gorey,2
5,COOKER HOODS,HLTHDS110SL/,RANGEMASTER 110CM COOKER HOOD,Gorey,1
6,COOKERS,LKR555100X,Electrolux 55cm St/St Cooker,Gorey,10
7,COOKERS,LKR555100B,Electrolux 55cm Black Cooker,Gorey,8
8,COOKERS,LKR655210X,Electrolux 60cm St/St Cooker,Gorey,4
9,COOKERS,LKI655200X,Electrolux 60cm St/St Induction Cooker,Gorey,6


In [13]:
## Validate long-format structure
expected_long_rows = (
    len(stock_raw) * len(expected_store_columns)
)

print("Expected rows :", expected_long_rows)
print("Actual rows   :", len(stock_long))
print("Row check     :", len(stock_long) == expected_long_rows)

print("\nColumns:")
print(stock_long.columns.tolist())

print("\nMissing values:")
display(
    stock_long.isna()
    .sum()
    .to_frame("missing_count")
)

Expected rows : 2681
Actual rows   : 2681
Row check     : True

Columns:
['Category', 'Model', 'Description', 'Store', 'Quantity']

Missing values:


,missing_count
Category,0
Model,0
Description,0
Store,0
Quantity,0


In [14]:
## Check product-store uniqueness
duplicate_product_store = stock_long.duplicated(
    subset=["Model", "Store"]
).sum()

unique_models_long = stock_long["Model"].nunique()
unique_stores_long = stock_long["Store"].nunique()

print("Unique models             :", unique_models_long)
print("Unique stores             :", unique_stores_long)
print("Duplicate Model-Store keys:", duplicate_product_store)

print("\nStores:")
print(sorted(stock_long["Store"].unique()))

Unique models             : 383
Unique stores             : 7
Duplicate Model-Store keys: 0

Stores:
['Belfast', 'Blanch', 'Cavan', 'Dundrum', 'Gorey', 'Navan', 'Sandyford']


In [15]:
## Critical post-reshape reconciliation
source_total = stock_raw["Total"].sum()
long_total = stock_long["Quantity"].sum()

print("Source stock total   :", source_total)
print("Reshaped stock total :", long_total)
print("Difference           :", long_total - source_total)
print("Reconciled           :", long_total == source_total)

Source stock total   : 6001
Reshaped stock total : 6001
Difference           : 0
Reconciled           : True


In [16]:
## Store-level reconciliation
source_store_totals = (
    stock_raw[expected_store_columns]
    .sum()
    .rename("Source_Total")
)

long_store_totals = (
    stock_long
    .groupby("Store")["Quantity"]
    .sum()
    .rename("Long_Total")
)

store_reconciliation = pd.concat(
    [source_store_totals, long_store_totals],
    axis=1
)

store_reconciliation["Difference"] = (
    store_reconciliation["Long_Total"]
    - store_reconciliation["Source_Total"]
)

store_reconciliation["Reconciled"] = (
    store_reconciliation["Difference"] == 0
)

display(store_reconciliation)

,Source_Total,Long_Total,Difference,Reconciled
Gorey,2850,2850,0,True
Dundrum,293,293,0,True
Cavan,1094,1094,0,True
Navan,234,234,0,True
Blanch,871,871,0,True
Sandyford,254,254,0,True
Belfast,405,405,0,True


#### Step 2.2 — Negative Stock Investigation & DQ Flag

In [17]:
## Inspect all negative records
negative_stock = (
    stock_long.loc[
        stock_long["Quantity"] < 0
    ]
    .sort_values(["Store", "Quantity"])
    .reset_index(drop=True)
)

print("Negative stock records:", len(negative_stock))

display(negative_stock)

Negative stock records: 12


,Category,Model,Description,Store,Quantity
0,HOBS,P154MDTC,Powerpoint Touch Control Ceramic Hob,Blanch,-1
1,U/C FRIDGE,P455LM3W,Powerpoint 55cm U/C Larder Fridge,Blanch,-1
2,WASHING MACHINES,IMA764MYTIMEUK,Indesit 7kg 1400 Spin Washing Machine,Blanch,-1
3,MICROWAVE OVENS,980533,DIMPLEX 20 LITRE 800 WATT BLACK MANUAL,Gorey,-2
4,SINGLE OVENS,B54CR71G0B,Neff N70 Graphite Single Oven,Gorey,-1
5,HOBS,PKE611CA3E,*Bosch 60cm Serie 2 Ceramic Hob with Knobs,Navan,-1
6,INT DISHWASHERS,S187ZCX03G,Neff N70 Integrated Dishwasher,Navan,-1
7,INT FREEZERS,WHSD18F023C1,Whirlpool Integrated Tall Freezer,Navan,-1
8,INT FRIDGES,WHSD18A033C1,Whirlpool Tall Integrated Larder Fridge,Navan,-1
9,TUMBLE DRYERS,DV90DB8845GBU1,Samsung Series 8 9kg Heat Pump Dryer,Navan,-1


In [18]:
## Negative stock summary
negative_stock_summary = (
    negative_stock
    .groupby("Store")
    .agg(
        Negative_Records=("Quantity", "count"),
        Negative_Quantity=("Quantity", "sum")
    )
    .sort_values(
        "Negative_Records",
        ascending=False
    )
)

display(negative_stock_summary)

,Negative_Records,Negative_Quantity
Store,,
Navan,6,-6
Blanch,3,-3
Gorey,2,-3
Sandyford,1,-1


In [19]:
# Business interpretation of stock quantity
stock_long["Stock_Status"] = np.select(
    [
        stock_long["Quantity"] < 0,
        stock_long["Quantity"] == 0,
        stock_long["Quantity"] > 0
    ],
    [
        "CUSTOMER_ORDER_OUTSTANDING",
        "OUT_OF_STOCK",
        "IN_STOCK"
    ],
    default="UNKNOWN"
)

# Number of units committed to customers but not currently available
stock_long["Outstanding_Order_Qty"] = np.where(
    stock_long["Quantity"] < 0,
    stock_long["Quantity"].abs(),
    0
)

print("Stock status distribution:")
display(
    stock_long["Stock_Status"]
    .value_counts()
    .to_frame("Record_Count")
)

print("\nOutstanding customer-order quantity:")
print(stock_long["Outstanding_Order_Qty"].sum())

Stock status distribution:


,Record_Count
Stock_Status,
IN_STOCK,1876
OUT_OF_STOCK,793
CUSTOMER_ORDER_OUTSTANDING,12



Outstanding customer-order quantity:
13


In [20]:
## DQ_Negative_Stock_Flag
print("Rows                         :", len(stock_long))
print("Stock total                  :", stock_long["Quantity"].sum())

print(
    "Customer-order-outstanding records:",
    (stock_long["Stock_Status"] == "CUSTOMER_ORDER_OUTSTANDING").sum()
)

print(
    "Outstanding customer units   :",
    stock_long["Outstanding_Order_Qty"].sum()
)

print(
    "Duplicate Model-Store keys    :",
    stock_long.duplicated(
        subset=["Model", "Store"]
    ).sum()
)

print(
    "Missing Quantity              :",
    stock_long["Quantity"].isna().sum()
)

Rows                         : 2681
Stock total                  : 6001
Customer-order-outstanding records: 12
Outstanding customer units   : 13
Duplicate Model-Store keys    : 0
Missing Quantity              : 0


In [21]:
validation_results = {
    "Expected rows": len(stock_raw) * len(expected_store_columns),
    "Actual rows": len(stock_long),
    "Unique models": stock_long["Model"].nunique(),
    "Unique stores": stock_long["Store"].nunique(),
    "Duplicate Model-Store keys": stock_long.duplicated(
        subset=["Model", "Store"]
    ).sum(),
    "Missing quantities": stock_long["Quantity"].isna().sum(),
    "Source stock total": stock_raw["Total"].sum(),
    "Governed stock total": stock_long["Quantity"].sum(),
    "Outstanding-order records": (
        stock_long["Stock_Status"] == "CUSTOMER_ORDER_OUTSTANDING"
    ).sum(),
    "Outstanding customer units": stock_long["Outstanding_Order_Qty"].sum()
}

validation_df = pd.DataFrame(
    validation_results.items(),
    columns=["Validation_Check", "Result"]
)

display(validation_df)

,Validation_Check,Result
0,Expected rows,2681
1,Actual rows,2681
2,Unique models,383
3,Unique stores,7
4,Duplicate Model-Store keys,0
5,Missing quantities,0
6,Source stock total,6001
7,Governed stock total,6001
8,Outstanding-order records,12
9,Outstanding customer units,13


In [22]:
reshape_gate_passed = (
    len(stock_long) == 2681
    and stock_long["Model"].nunique() == 383
    and stock_long["Store"].nunique() == 7
    and stock_long.duplicated(
        subset=["Model", "Store"]
    ).sum() == 0
    and stock_long["Quantity"].isna().sum() == 0
    and stock_long["Quantity"].sum() == stock_raw["Total"].sum()
)

print(
    "STEP 2.1 RESHAPE GATE:",
    "PASSED" if reshape_gate_passed else "FAILED"
)

STEP 2.1 RESHAPE GATE: PASSED


### Step 3 — Create governed fact_stock

In [23]:
fact_stock = stock_long[
    [
        "Model",
        "Store",
        "Category",
        "Description",
        "Quantity",
        "Stock_Status",
        "Outstanding_Order_Qty"
    ]
].copy()

fact_stock = fact_stock.sort_values(
    ["Model", "Store"]
).reset_index(drop=True)

print("fact_stock shape:", fact_stock.shape)

display(fact_stock.head(10))

fact_stock shape: (2681, 7)


,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty
0,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
1,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
2,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
3,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0
4,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0
5,980531,Navan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
6,980531,Sandyford,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
7,980532,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,1,IN_STOCK,0
8,980532,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,4,IN_STOCK,0
9,980532,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,2,IN_STOCK,0


In [ ]:

# ============================================================
# ADD GOVERNED PRODUCT KEYS TO fact_stock
# ============================================================
DIM_PRODUCT_PATH = PROJECT_ROOT / "data" / "processed" / "dim_product.csv"
dim_product = pd.read_csv(DIM_PRODUCT_PATH )
# Build mapping from dim_product
stock_product_map = (
    dim_product[
        ["Product_ID","Stock_Model"]
    ]
    .dropna(subset=["Stock_Model"])
    .copy()
)

# Standardise keys
stock_product_map["Stock_Model"] = (
    stock_product_map["Stock_Model"]
    .astype(str)
    .str.strip()
    .str.upper()
)

fact_stock["Model"] = (
    fact_stock["Model"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Ensure one Stock_Model maps to one product
if stock_product_map["Stock_Model"].duplicated().any():
    raise ValueError(
        "Duplicate Stock_Model found in dim_product."
    )

# Attach Product_ID 
fact_stock = fact_stock.merge(
    stock_product_map,
    left_on="Model",
    right_on="Stock_Model",
    how="left",
    validate="many_to_one"
)

# Bridge column no longer required
fact_stock = fact_stock.drop(columns=["Stock_Model"])

# Reorder columns
fact_stock = fact_stock[
    [
        "Product_ID",
        "Model",
        "Store",
        "Category",
        "Description",
        "Quantity",
        "Stock_Status",
        "Outstanding_Order_Qty"
    ]
]

# ============================================================
# VALIDATION
# ============================================================

print("UPDATED FACT_STOCK")
print("=" * 80)

print(f"Rows                : {len(fact_stock):,}")
print(f"Columns             : {fact_stock.shape[1]}")
print(f"Unique models       : {fact_stock['Model'].nunique():,}")
print(f"Unique Product_IDs  : {fact_stock['Product_ID'].nunique():,}")
print(f"Missing Product_IDs : {fact_stock['Product_ID'].isna().sum():,}")

print()
print("VALIDATION")
print("=" * 80)

print(
    "All models mapped:",
    fact_stock["Product_ID"].isna().sum() == 0
)

print(
    "Model → Product_ID one-to-one:",
    fact_stock[
        ["Model", "Product_ID"]
    ].drop_duplicates()["Model"].nunique()
    ==
    fact_stock["Model"].nunique()
)

display(fact_stock.head(10))

UPDATED FACT_STOCK
Rows                : 2,681
Columns             : 8
Unique models       : 383
Unique Product_IDs  : 383
Missing Product_IDs : 0

VALIDATION
All models mapped: True
Model → Product_ID one-to-one: True


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0
5,221,980531,Navan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
6,221,980531,Sandyford,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
7,222,980532,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,1,IN_STOCK,0
8,222,980532,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,4,IN_STOCK,0
9,222,980532,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,2,IN_STOCK,0


In [ ]:
## Save the file

""" fact_stock.to_csv(
    "../../data/processed/fact_stock.csv",
    index=False
)

print("fact_stock.csv saved successfully.") """

fact_stock.csv saved successfully.
